In [1]:
import cobra

model = cobra.io.read_sbml_model("../data/iJN1463.xml")
print(model.id)
print("reactions  ", len(model.reactions))
print("metabolites", len(model.metabolites))
print("genes      ", len(model.genes))
print("objective  ", model.objective.expression)

iJN1463
reactions   2929
metabolites 2155
genes       1462
objective   1.0*BIOMASS_KT2440_WT3 - 1.0*BIOMASS_KT2440_WT3_reverse_d86d5


In [2]:
sol = model.optimize()
print("status:", sol.status)
print("growth:", round(sol.objective_value, 5), "/h")

print("\ndefault medium:")
for r, b in sorted(model.medium.items()):
    print(f"  {r:<28} {b}")

status: optimal
growth: 0.58612 /h

default medium:
  EX___iJN1463__PHAg_c         999999.0
  EX___iJN1463__acmtsoxin_e    1000.0
  EX___iJN1463__acpptrn_e      1000.0
  EX___iJN1463__d2one_e        1000.0
  EX___iJN1463__d3one_e        1000.0
  EX___iJN1463__d4one_e        1000.0
  EX___iJN1463__mtsoxin_e      1000.0
  EX___iJN1463__n2one_e        1000.0
  EX___iJN1463__pptrn_e        1000.0
  EX___iJN1463__pqqA_kt_c      1.0
  EX___iJN1463__und2one_e      1000.0
  EX_ca2_e                     10.0
  EX_cl_e                      10.0
  EX_co2_e                     100.0
  EX_cobalt2_e                 10.0
  EX_cu2_e                     10.0
  EX_fe2_e                     10.0
  EX_glc__D_e                  6.0
  EX_h2co3_e                   10.0
  EX_h2o_e                     100.0
  EX_h_e                       100.0
  EX_k_e                       10.0
  EX_mg2_e                     10.0
  EX_mn2_e                     10.0
  EX_mobd_e                    10.0
  EX_na1_e               

In [3]:
# succinate dehydrogenase: what does it hand electrons to?
sdh = model.reactions.get_by_id("SUCDi")
print(sdh.name)
print(" ", sdh.reaction)
print("  genes:", sdh.gene_reaction_rule)

# the first EG oxidation step
eg = model.reactions.get_by_id("__iJN1463__GLYCOLDHpp")
print("\n" + eg.name)
print(" ", eg.reaction)
print("  genes:", eg.gene_reaction_rule)

# what reoxidises the PQQH2 that step produces?
print("\nPQQH2 consumers:")
pqqh2 = model.metabolites.get_by_id("pqqh2_p")
for r in pqqh2.reactions:
    if r.metabolites[pqqh2] < 0:
        print(f"  {r.id:<26} {r.reaction}")

succinate dehydrogenase (irreversible)
  q8_c + succ_c --> fum_c + q8h2_c
  genes: PP_4190 and PP_4191 and PP_4192 and PP_4193

R_glycol_dehydrogenase__forward_rxn__ethanol___acetaldehyde_
  __iJN1463__glycol_p + pqq_p --> gcald_p + pqqh2_p
  genes: PP_2674 or PP_2679

PQQH2 consumers:
  __iJN1463__FE3PYOVDDR2     2.0 __iJN1463__fe3pyovd_kt_p + pqqh2_p <=> 2.0 __iJN1463__pyovd_kt_p + 2.0 fe2_p + 2.0 h_p + pqq_p
  __iJN1463__CYO4pp          2.0 __iJN1463__ficytc_c + pqqh2_p --> 2.0 __iJN1463__focytc_c + pqq_p


In [4]:
# step 2 of the EG route
print(model.reactions.get_by_id("GCALDD").reaction)

# what reduces the quinone pool?
q8 = model.metabolites.get_by_id("q8_c")
print("\nreactions that consume q8 (i.e. reduce it to q8h2):")
for r in q8.reactions:
    if r.metabolites[q8] < 0:
        print(f"  {r.id:<16} {r.reaction}")

gcald_c + h2o_c + nad_c --> glyclt_c + 2.0 h_c + nadh_c

reactions that consume q8 (i.e. reduce it to q8h2):
  G3PD5            glyc3p_c + q8_c --> dhap_c + q8h2_c
  NADPHQR2         h_c + nadph_c + q8_c --> nadp_c + q8h2_c
  LDH_D2           lac__D_c + q8_c --> pyr_c + q8h2_c
  MDH2             mal__L_c + q8_c --> oaa_c + q8h2_c
  NADH16pp         4.0 h_c + nadh_c + q8_c --> 3.0 h_p + nad_c + q8h2_c
  ASPO3            asp__L_c + q8_c --> h_c + iasp_c + q8h2_c
  GLCDpp           glc__D_p + h2o_p + q8_c --> glcn_p + h_p + q8h2_c
  NADH5            h_c + nadh_c + q8_c --> nad_c + q8h2_c
  __iJN1463__SUCD4 fadh2_c + q8_c <=> __iJN1463__fad_c + q8h2_c
  __iJN1463__DSBAO1 __iJN1463__dsbard_p + q8_c --> __iJN1463__dsbaox_p + q8h2_c
  SUCDi            q8_c + succ_c --> fum_c + q8h2_c
  GAD2ktpp         glcn_p + q8_c <=> 2dhglcn_p + q8h2_c
  DHORD2           dhor__S_c + q8_c --> orot_c + q8h2_c
  L_LACD2          lac__L_c + q8_c --> pyr_c + q8h2_c


In [5]:
# NADH produced per unit substrate, each route
for ex_id, label in [("EX___iJN1463__glycol_e", "glycol"),
                     ("EX___iJN1463__34dhbz_e", "protocatechuate")]:
    with model:
        medium = model.medium
        medium["EX_glc__D_e"] = 0.0
        medium[ex_id] = 10.0
        model.medium = medium
        s = model.optimize()
        nadh = sum(s.fluxes[r.id] * r.metabolites[model.metabolites.get_by_id("nadh_c")]
                   for r in model.metabolites.get_by_id("nadh_c").reactions)
        print(f"{label:<18} growth {s.objective_value:.4f}")

glycol             growth 0.2381
protocatechuate    growth 0.8577


In [6]:
nadh_c = model.metabolites.get_by_id("nadh_c")

def audit(ex_id, label, uptake=10.0):
    met_id = ex_id.replace("EX_", "")
    carbons = model.metabolites.get_by_id(met_id).elements.get("C", 0)
    with model:
        medium = model.medium
        medium["EX_glc__D_e"] = 0.0
        medium[ex_id] = uptake
        model.medium = medium
        s = model.optimize()
        made = sum(s.fluxes[r.id] * r.metabolites[nadh_c]
                   for r in nadh_c.reactions
                   if s.fluxes[r.id] * r.metabolites[nadh_c] > 0)
        totalC = uptake * carbons
        print(f"{label:<18} C{carbons}  growth {s.objective_value:.4f}  "
              f"NADH {made:7.2f}  NADH/C {made/totalC:.3f}")

audit("EX___iJN1463__glycol_e", "glycol")
audit("EX___iJN1463__34dhbz_e", "protocatechuate")
audit("EX_glc__D_e", "glucose")

glycol             C2  growth 0.2381  NADH   18.47  NADH/C 0.923
protocatechuate    C7  growth 0.8577  NADH   24.93  NADH/C 0.356
glucose            C6  growth 0.9796  NADH   23.57  NADH/C 0.393


In [7]:
from cobra.flux_analysis import flux_variability_analysis

nadh_c = model.metabolites.get_by_id("nadh_c")

SUBS = [("EX___iJN1463__glycol_e", "glycol"),
        ("EX___iJN1463__34dhbz_e", "protocatechuate"),
        ("EX_glc__D_e", "glucose")]

CARBON = 70.0   # feed every substrate the same amount of carbon

print(f"{'substrate':<18} {'uptake':>7} {'growth':>8} {'NADH/C':>8} {'O2/C':>7}")
print("-" * 52)

for ex_id, label in SUBS:
    met = model.metabolites.get_by_id(ex_id.replace("EX_", ""))
    carbons = met.elements.get("C", 0)
    uptake = CARBON / carbons          # same carbon in, whatever the molecule

    with model:
        medium = model.medium
        medium["EX_glc__D_e"] = 0.0
        medium[ex_id] = uptake
        model.medium = medium

        s = model.optimize()
        nadh = sum(s.fluxes[r.id] * r.metabolites[nadh_c]
                   for r in nadh_c.reactions
                   if s.fluxes[r.id] * r.metabolites[nadh_c] > 0)
        o2 = -s.fluxes["EX_o2_e"]

        print(f"{label:<18} {uptake:7.2f} {s.objective_value:8.4f} "
              f"{nadh/CARBON:8.3f} {o2/CARBON:7.3f}")

substrate           uptake   growth   NADH/C    O2/C
----------------------------------------------------
glycol               35.00   0.8435    1.162   0.745
protocatechuate      10.00   0.8577    0.406   0.415
glucose              11.67   1.1436    0.396   0.315


In [8]:
# the single numbers above are one of many possible answers.
# this shows the lowest and highest each reaction could run at.

for ex_id, label in SUBS:
    met = model.metabolites.get_by_id(ex_id.replace("EX_", ""))
    uptake = CARBON / met.elements.get("C", 0)

    with model:
        medium = model.medium
        medium["EX_glc__D_e"] = 0.0
        medium[ex_id] = uptake
        model.medium = medium

        fva = flux_variability_analysis(
            model,
            reaction_list=["NADH16pp", "NADH5", "SUCDi", "GCALDD"],
            fraction_of_optimum=0.9)

        print(f"\n--- {label} ---")
        print(fva.round(2).to_string())


--- glycol ---
          minimum  maximum
NADH16pp    11.11    41.32
NADH5        0.00    24.96
SUCDi        0.00    30.21
GCALDD      31.52    35.00

--- protocatechuate ---
          minimum  maximum
NADH16pp     1.55    35.95
NADH5        0.00    25.21
SUCDi        9.04    43.71
GCALDD       0.00     0.00

--- glucose ---
          minimum  maximum
NADH16pp     0.83    53.67
NADH5        0.00    33.71
SUCDi        0.00    41.49
GCALDD       0.00     0.00
